# Sprint 2B: Model Development

This notebook uses the saved feature-engineering output to train and compare models for both vaccination targets.

**Input:** `storage/train_engineered.csv`  
**Outputs:** trained pipelines in `models/` and performance tables in `results/`

The modelling workflow is:

1. Load and validate the engineered dataset.
2. Separate predictors and targets.
3. Split training and validation data.
4. Build the preprocessing pipeline.
5. Define baseline models.
6. Apply cross-validation.
7. Tune hyperparameters.
8. Evaluate tuned models.
9. Interpret important predictors.
10. Save the models and result tables.

## STAGE 0: Import packages and create output folders

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from joblib import dump

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    cross_validate,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

RANDOM_STATE = 42

STORAGE_DIR = Path("../storage")
MODEL_DIR = Path("../models")
RESULTS_DIR = Path("../results")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

## STAGE 1: Load and validate the engineered dataset

The file should contain the respondent ID, 35 original predictors, 12 engineered predictors and two targets: 50 columns in total.

In [ ]:
engineered_data_path = (
    STORAGE_DIR / "train_engineered.csv"
)

model_data_df = pd.read_csv(
    engineered_data_path
)

print("Loaded:", engineered_data_path)
print("Engineered data shape:", model_data_df.shape)
display(model_data_df.head())

In [ ]:
target_columns = [
    "h1n1_vaccine",
    "seasonal_vaccine"
]

required_columns = [
    "respondent_id",
    *target_columns,
    "preventive_behavior_score",
    "h1n1_risk_perception_score",
    "h1n1_vaccine_confidence_score",
    "seasonal_vaccine_confidence_score",
    "household_size",
    "any_doctor_recommendation"
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in model_data_df.columns
]

print("Missing required columns:", missing_required_columns)
print("Duplicate respondent IDs:", model_data_df["respondent_id"].duplicated().sum())
print("Missing target values:")
print(model_data_df[target_columns].isna().sum())

if missing_required_columns:
    raise ValueError(
        f"Required modelling columns are missing: {missing_required_columns}"
    )

if model_data_df["respondent_id"].duplicated().any():
    raise ValueError("Duplicate respondent IDs were found.")

if model_data_df[target_columns].isna().any().any():
    raise ValueError("Missing target values were found.")

## STAGE 2: Separate predictors and target variables

`X` contains the original and engineered predictors. The respondent ID and the two outcomes are excluded. Each outcome is stored as a separate target because this project trains a separate classifier for each vaccine.

In [ ]:
X = model_data_df.drop(
    columns=[
        "respondent_id",
        "h1n1_vaccine",
        "seasonal_vaccine"
    ]
)

y_h1n1 = model_data_df["h1n1_vaccine"].astype(int)
y_seasonal = model_data_df["seasonal_vaccine"].astype(int)

print("Predictor data:", X.shape)
print("H1N1 target:", y_h1n1.shape)
print("Seasonal target:", y_seasonal.shape)
print("Targets accidentally present in X:", set(target_columns).intersection(X.columns))

## STAGE 3: Split the training and validation data

- 80% of respondents are used for training.
- 20% are kept untouched for final validation.
- Stratification preserves the four combinations of the two vaccination outcomes.

In [ ]:
combined_target = (
    y_h1n1.astype(str)
    + "_"
    + y_seasonal.astype(str)
)

(
    X_train,
    X_valid,
    y_h1n1_train,
    y_h1n1_valid,
    y_seasonal_train,
    y_seasonal_valid
) = train_test_split(
    X,
    y_h1n1,
    y_seasonal,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=combined_target
)

print("Training predictors:", X_train.shape)
print("Validation predictors:", X_valid.shape)
print("H1N1 training target:", y_h1n1_train.shape)
print("Seasonal training target:", y_seasonal_train.shape)

## STAGE 4: Build the preprocessing pipeline

Each predictor type needs different preparation:

- numeric: median imputation and standardization;
- ordinal: most-common imputation and order-preserving encoding;
- nominal: `Unknown` imputation and one-hot encoding.

The preprocessing rules are fitted only on training data when a model pipeline is trained.

In [ ]:
# Ordered categorical predictors.
ordinal_columns = [
    "age_group",
    "education",
    "income_poverty"
]

# Unordered categorical predictors.
nominal_columns = [
    "race",
    "sex",
    "marital_status",
    "rent_or_own",
    "employment_status",
    "census_msa",
    "hhs_geo_region",
    "employment_industry",
    "employment_occupation"
]

# Every remaining predictor is numeric.
numeric_columns = [
    column
    for column in X.columns
    if column not in ordinal_columns + nominal_columns
]

print("Numeric columns:", len(numeric_columns))
print("Ordinal columns:", len(ordinal_columns))
print("Nominal columns:", len(nominal_columns))
print("Total grouped columns:", len(numeric_columns + ordinal_columns + nominal_columns))
print("Total X columns:", X.shape[1])

In [ ]:
# Numeric rules: fill missing values using the training median, then scale.
numeric_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# Correct order for the three ordinal predictors.
age_order = [
    "18 - 34 Years",
    "35 - 44 Years",
    "45 - 54 Years",
    "55 - 64 Years",
    "65+ Years"
]

education_order = [
    "< 12 Years",
    "12 Years",
    "Some College",
    "College Graduate"
]

income_order = [
    "Below Poverty",
    "<= $75,000, Above Poverty",
    "> $75,000"
]

# Ordinal rules: fill missing categories, then preserve their natural order.
ordinal_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OrdinalEncoder(
                categories=[
                    age_order,
                    education_order,
                    income_order
                ],
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )
        )
    ]
)

# Nominal rules: create an Unknown category, then one-hot encode.
# Dense output works with all three selected model families.
nominal_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

# Send each group through its own transformation rules.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_preprocessor,
            numeric_columns
        ),
        (
            "ordinal",
            ordinal_preprocessor,
            ordinal_columns
        ),
        (
            "nominal",
            nominal_preprocessor,
            nominal_columns
        )
    ]
)

print("Preprocessor created successfully.")

## STAGE 5: Define the baseline classification models

Three model families are compared:

- Logistic Regression: interpretable linear baseline;
- Random Forest: captures non-linear patterns and interactions;
- Gradient Boosting: builds a sequence of trees that correct earlier errors.

Each model is attached to its own copy of the preprocessor.

In [ ]:
baseline_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    )
}


def make_model_pipeline(model):
    # Combine an independent preprocessor with one classifier.

    return Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", clone(model))
        ]
    )


print("Baseline models:", list(baseline_models))

## STAGE 6: Cross-validation on the training data

Cross-validation divides the training portion into several temporary folds. Each model trains and validates repeatedly, producing a more stable performance estimate than a single split.

The untouched 20% validation set is not used here.

In [ ]:
scoring_metrics = {
    "roc_auc": "roc_auc",
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

training_targets = {
    "H1N1": y_h1n1_train,
    "Seasonal": y_seasonal_train
}

cv_rows = []

for target_name, y_train in training_targets.items():
    for model_name, model in baseline_models.items():
        model_pipeline = make_model_pipeline(model)

        cv_scores = cross_validate(
            model_pipeline,
            X_train,
            y_train,
            cv=5,
            scoring=scoring_metrics,
            n_jobs=-1
        )

        cv_rows.append(
            {
                "Target": target_name,
                "Model": model_name,
                "CV ROC-AUC": cv_scores["test_roc_auc"].mean(),
                "CV Accuracy": cv_scores["test_accuracy"].mean(),
                "CV Precision": cv_scores["test_precision"].mean(),
                "CV Recall": cv_scores["test_recall"].mean(),
                "CV F1": cv_scores["test_f1"].mean()
            }
        )

        print(f"Completed: {target_name} - {model_name}")

cv_results_df = (
    pd.DataFrame(cv_rows)
    .sort_values(["Target", "CV ROC-AUC"], ascending=[True, False])
    .reset_index(drop=True)
)

display(cv_results_df.round(3))

In [ ]:
# Compare baseline ROC-AUC scores visually.
plt.figure(figsize=(10, 5))

sns.barplot(
    data=cv_results_df,
    x="Model",
    y="CV ROC-AUC",
    hue="Target"
)

plt.title("Baseline Model Comparison: Cross-Validated ROC-AUC")
plt.ylim(0.5, 1.0)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## STAGE 7: Hyperparameter tuning

Hyperparameters are model settings chosen before training. `RandomizedSearchCV` tries a small selection of settings using three-fold cross-validation and selects the combination with the highest ROC-AUC.

This notebook tunes all three model families for both targets. Reduce `N_TUNING_ITERATIONS` if the computer takes too long.

In [ ]:
parameter_distributions = {
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10, 100],
        "model__class_weight": [None, "balanced"]
    },
    "Random Forest": {
        "model__n_estimators": [150, 250, 400],
        "model__max_depth": [None, 8, 15, 25],
        "model__min_samples_leaf": [1, 2, 5, 10],
        "model__max_features": ["sqrt", "log2"],
        "model__class_weight": [None, "balanced"]
    },
    "Gradient Boosting": {
        "model__n_estimators": [75, 100, 150, 200],
        "model__learning_rate": [0.03, 0.05, 0.1, 0.2],
        "model__max_depth": [1, 2, 3],
        "model__min_samples_leaf": [1, 5, 10]
    }
}

N_TUNING_ITERATIONS = 8

best_models = {}
tuning_rows = []

for target_name, y_train in training_targets.items():
    best_models[target_name] = {}

    for model_name, model in baseline_models.items():
        search = RandomizedSearchCV(
            estimator=make_model_pipeline(model),
            param_distributions=parameter_distributions[model_name],
            n_iter=N_TUNING_ITERATIONS,
            scoring="roc_auc",
            cv=3,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            refit=True
        )

        search.fit(X_train, y_train)

        best_models[target_name][model_name] = search.best_estimator_

        tuning_rows.append(
            {
                "Target": target_name,
                "Model": model_name,
                "Best tuning ROC-AUC": search.best_score_,
                "Best parameters": search.best_params_
            }
        )

        print(f"Tuned: {target_name} - {model_name}")

tuning_results_df = (
    pd.DataFrame(tuning_rows)
    .sort_values(
        ["Target", "Best tuning ROC-AUC"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(tuning_results_df)

## STAGE 8: Evaluate tuned models on the untouched validation data

- Class predictions use a default probability threshold of 0.50.
- ROC-AUC uses predicted probabilities and is the primary comparison metric.
- Accuracy, precision, recall and F1 describe different aspects of classification performance.

In [ ]:
validation_targets = {
    "H1N1": y_h1n1_valid,
    "Seasonal": y_seasonal_valid
}

validation_rows = []

for target_name, y_valid in validation_targets.items():
    for model_name, fitted_pipeline in best_models[target_name].items():
        predicted_class = fitted_pipeline.predict(X_valid)
        predicted_probability = fitted_pipeline.predict_proba(X_valid)[:, 1]

        validation_rows.append(
            {
                "Target": target_name,
                "Model": model_name,
                "ROC-AUC": roc_auc_score(y_valid, predicted_probability),
                "Accuracy": accuracy_score(y_valid, predicted_class),
                "Precision": precision_score(y_valid, predicted_class, zero_division=0),
                "Recall": recall_score(y_valid, predicted_class, zero_division=0),
                "F1": f1_score(y_valid, predicted_class, zero_division=0)
            }
        )

validation_results_df = (
    pd.DataFrame(validation_rows)
    .sort_values(["Target", "ROC-AUC"], ascending=[True, False])
    .reset_index(drop=True)
)

display(validation_results_df.round(3))

In [ ]:
# Identify the best validation model for each target using ROC-AUC.
best_model_names = (
    validation_results_df
    .loc[
        validation_results_df.groupby("Target")["ROC-AUC"].idxmax(),
        ["Target", "Model", "ROC-AUC"]
    ]
    .reset_index(drop=True)
)

display(best_model_names.round(3))

### 8.1 Confusion matrices for the best models

The confusion matrix shows true negatives, false positives, false negatives and true positives at the 0.50 threshold.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for axis, target_name in zip(axes, ["H1N1", "Seasonal"]):
    model_name = best_model_names.loc[
        best_model_names["Target"].eq(target_name),
        "Model"
    ].iloc[0]

    fitted_pipeline = best_models[target_name][model_name]
    y_valid = validation_targets[target_name]
    predicted_class = fitted_pipeline.predict(X_valid)

    ConfusionMatrixDisplay.from_predictions(
        y_valid,
        predicted_class,
        ax=axis,
        cmap="Blues",
        colorbar=False
    )

    axis.set_title(f"{target_name}: {model_name}")

plt.tight_layout()
plt.show()

### 8.2 ROC curves for the best models

The ROC curve examines model performance across all possible classification thresholds.

In [ ]:
fig, axis = plt.subplots(figsize=(7, 6))

for target_name in ["H1N1", "Seasonal"]:
    model_name = best_model_names.loc[
        best_model_names["Target"].eq(target_name),
        "Model"
    ].iloc[0]

    fitted_pipeline = best_models[target_name][model_name]

    RocCurveDisplay.from_estimator(
        fitted_pipeline,
        X_valid,
        validation_targets[target_name],
        name=f"{target_name} - {model_name}",
        ax=axis
    )

axis.plot([0, 1], [0, 1], linestyle="--", color="grey")
axis.set_title("ROC Curves for the Best Validation Models")
plt.tight_layout()
plt.show()

### 8.3 Detailed classification reports

These reports show class-level precision, recall and F1. Class `1` represents vaccinated respondents.

In [ ]:
for target_name in ["H1N1", "Seasonal"]:
    model_name = best_model_names.loc[
        best_model_names["Target"].eq(target_name),
        "Model"
    ].iloc[0]

    fitted_pipeline = best_models[target_name][model_name]
    predicted_class = fitted_pipeline.predict(X_valid)

    print(f"\n{target_name} - {model_name}")
    print(
        classification_report(
            validation_targets[target_name],
            predicted_class,
            digits=3,
            zero_division=0
        )
    )

## STAGE 9: Interpret the best models

Permutation importance measures how much validation ROC-AUC decreases when one original predictor is randomly shuffled. A larger decrease suggests that the fitted pipeline relies more strongly on that predictor.

Importance is an association with prediction, not proof that the feature causes vaccination.

In [ ]:
importance_tables = {}

for target_name in ["H1N1", "Seasonal"]:
    model_name = best_model_names.loc[
        best_model_names["Target"].eq(target_name),
        "Model"
    ].iloc[0]

    fitted_pipeline = best_models[target_name][model_name]

    importance_result = permutation_importance(
        fitted_pipeline,
        X_valid,
        validation_targets[target_name],
        scoring="roc_auc",
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    importance_df = (
        pd.DataFrame(
            {
                "Feature": X_valid.columns,
                "Importance": importance_result.importances_mean,
                "Importance SD": importance_result.importances_std
            }
        )
        .sort_values("Importance", ascending=False)
        .reset_index(drop=True)
    )

    importance_tables[target_name] = importance_df

    print(f"\nTop predictors for {target_name} ({model_name})")
    display(importance_df.head(15).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for axis, target_name in zip(axes, ["H1N1", "Seasonal"]):
    top_features = importance_tables[target_name].head(15).sort_values("Importance")

    axis.barh(
        top_features["Feature"],
        top_features["Importance"],
        color="#2A6F8A" if target_name == "H1N1" else "#E58A3A"
    )

    axis.set_title(f"{target_name}: Top Predictive Features")
    axis.set_xlabel("Decrease in validation ROC-AUC after shuffling")

plt.tight_layout()
plt.show()

## STAGE 10: Save the Sprint 2 deliverables

The best fitted pipeline for each target includes both preprocessing and the selected classifier. Saving the complete pipeline ensures future data receives the same transformations before prediction.

In [ ]:
saved_model_paths = {}

for target_name in ["H1N1", "Seasonal"]:
    model_name = best_model_names.loc[
        best_model_names["Target"].eq(target_name),
        "Model"
    ].iloc[0]

    fitted_pipeline = best_models[target_name][model_name]

    model_path = MODEL_DIR / f"best_{target_name.lower()}_pipeline.joblib"
    dump(fitted_pipeline, model_path)
    saved_model_paths[target_name] = model_path

# Save performance tables and feature-importance tables.
cv_results_df.to_csv(
    RESULTS_DIR / "baseline_cross_validation_results.csv",
    index=False
)

tuning_results_df.to_csv(
    RESULTS_DIR / "hyperparameter_tuning_results.csv",
    index=False
)

validation_results_df.to_csv(
    RESULTS_DIR / "validation_model_comparison.csv",
    index=False
)

importance_tables["H1N1"].to_csv(
    RESULTS_DIR / "h1n1_feature_importance.csv",
    index=False
)

importance_tables["Seasonal"].to_csv(
    RESULTS_DIR / "seasonal_feature_importance.csv",
    index=False
)

print("Saved model pipelines:")
for target_name, path in saved_model_paths.items():
    print(f"- {target_name}: {path}")

print("\nSaved result tables in:", RESULTS_DIR)

## STAGE 11: Sprint 2 conclusions and presentation notes

Complete this section after running the notebook:

1. State the best H1N1 model and its validation ROC-AUC.
2. State the best seasonal model and its validation ROC-AUC.
3. Compare accuracy, precision, recall and F1.
4. Explain the most important predictors for each target.
5. Relate important predictors back to the Sprint 1 hypotheses.
6. Mention limitations:
   - feature importance does not prove causation;
   - survey answers are self-reported;
   - some predictors contain substantial missingness;
   - validation results estimate, but do not guarantee, performance on new populations.

### Suggested five-minute presentation

| Slide | Content |
|---|---|
| 1 | Problem, targets and Sprint 2 objective |
| 2 | Engineered features and preprocessing pipeline |
| 3 | Logistic Regression, Random Forest and Gradient Boosting comparison |
| 4 | Best ROC-AUC results and confusion matrices |
| 5 | Most important predictors, hypothesis findings and next steps |